In [1]:
#imports 
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [2]:
#loading the friday afternoon ddos dataset
print("Loading the DDoS dataset")
file_path = "MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

print("\n Traffic Distribution before cleaning: ")
print(df['Label'].value_counts())

Loading the DDoS dataset

 Traffic Distribution before cleaning: 
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [3]:
print("\n Cleaning and encoding:")

#we get rid of nans and infinites
df_clean = df.dropna()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

#encoding
df_clean['Label'] = df_clean['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print("\n Traffic Distribution after cleaning: ")
print(df_clean['Label'].value_counts())


 Cleaning and encoding:



 Traffic Distribution after cleaning: 
Label
1    128025
0     97686
Name: count, dtype: int64


In [4]:
#spitting and scaling

print("\n Splitting and Scaling Data")
X = df_clean.drop('Label', axis=1)
y = df_clean['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


 Splitting and Scaling Data


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) #explain in paper why fit transform and not simple fit
X_test_scaled = scaler.transform(X_test)

print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")

Training data shape: (180568, 78)
Testing data shape: (45143, 78)


In [6]:
#going over all the models again for accuracy

print("\n Multi-Model Cycling")

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),

    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, random_state=42)
}

results = []


 Multi-Model Cycling


In [7]:
for name, model in models.items():
    print(f"  -> Training {name}...")
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    train_time = time.time() - start_time
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Recall (Attack Catch Rate)": round(recall * 100, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 2)
    })

  -> Training Decision Tree...
  -> Training Random Forest...
  -> Training XGBoost...
  -> Training LightGBM...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Training Logistic Regression...
  -> Training MLP (Neural Net)...


In [8]:
print("\n Final DDoS model leaderboard")
results_df = pd.DataFrame(results).sort_values(by="Accuracy (%)", ascending=False)
display(results_df)


 Final DDoS model leaderboard


,Model,Accuracy (%),Recall (Attack Catch Rate),F1-Score,Training Time (s)
2,XGBoost,99.9978,100.0000,1.0000,4.65
3,LightGBM,99.9956,100.0000,1.0000,8.69
0,Decision Tree,99.9934,99.9922,0.9999,7.25
1,Random Forest,99.9889,99.9845,0.9999,7.78
5,MLP (Neural Net),99.9690,99.9806,0.9997,76.50
4,Logistic Regression,99.8693,99.8756,0.9989,9.23


This time we can see that compared to all the models XGBoost has performed the best with high accuracy and a low processing time (second best in time) (i can explain more in the paper why xgboost is a favorite in the industry when it comes to tabular data)

For the last one, the one that performed the worst, we have logistic regression. The accuracy might still be good in theory but compared to th rest it struggled. As a small explanation, logistic regression is the type to draw straight mathematical lines through the data, but network attacks are rarely linear 

Just like explained before, mlp took a lot of time. It did have good accuracy, but the amount of time it takes (even for low iterations) makes it hard to pick for these cases.

Between this and the port scans attack, we have the similarity that they are volumetric attacks. They generate loud and obvious traffic which is why my models score a high accuracy.
In this one and the port scan i didn t yet have imbalance, but for thursday attacks there will be